<a href="https://colab.research.google.com/github/Heymaah/CUDA/blob/main/quantization2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install torch torchvision matplotlib -q

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
import time
import os

import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms as transforms


In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cpu


In [6]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

testset = torchvision.datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=64,
    shuffle=True
)

testloader = torch.utils.data.DataLoader(
    testset,
    batch_size=64,
    shuffle=False
)

In [7]:
def evaluate_accuracy(model, loader):

    model.eval()

    device = next(model.parameters()).device

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total

In [8]:
def measure_latency(model, runs=50):

    model.eval()

    device = next(model.parameters()).device

    dummy = torch.randn(1,3,224,224).to(device)

    with torch.no_grad():
        for _ in range(10):
            model(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.time()

    with torch.no_grad():
        for _ in range(runs):
            model(dummy)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.time()

    return ((end-start)/runs)*1000

In [9]:
def model_size_mb(model, filename="temp.p"):

    torch.save(model.state_dict(), filename)

    size = os.path.getsize(filename)/1e6

    os.remove(filename)

    return size

In [10]:
fp32_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

num_features = fp32_model.fc.in_features

fp32_model.fc = nn.Linear(num_features,10)

fp32_model = fp32_model.to(DEVICE)

In [11]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    fp32_model.parameters(),
    lr=1e-4
)

EPOCHS = 1

for epoch in range(EPOCHS):

    fp32_model.train()

    for images, labels in trainloader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = fp32_model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

    print("Epoch", epoch+1, "completed")

Epoch 1 completed


In [ ]:
fp32_acc = evaluate_accuracy(
    fp32_model,
    testloader
)

fp32_lat = measure_latency(
    fp32_model
)

fp32_size = model_size_mb(
    fp32_model
)

print(fp32_acc, fp32_lat, fp32_size)

In [ ]:
ptq_dynamic = copy.deepcopy(
    fp32_model.cpu()
)

ptq_dynamic.eval()

ptq_dynamic = torch.quantization.quantize_dynamic(
    ptq_dynamic,
    {nn.Linear},
    dtype=torch.qint8
)

In [ ]:
ptq_dyn_acc = evaluate_accuracy(
    ptq_dynamic,
    testloader
)

ptq_dyn_lat = measure_latency(
    ptq_dynamic
)

ptq_dyn_size = model_size_mb(
    ptq_dynamic
)

In [ ]:
static_model = copy.deepcopy(
    fp32_model.cpu()
)

static_model.eval()

static_model.fuse_model = lambda : None

torch.quantization.fuse_modules(
    static_model,
    [
        ["conv1","bn1","relu"]
    ],
    inplace=True
)

In [ ]:
static_model.qconfig = (
    torch.quantization.get_default_qconfig("fbgemm")
)

torch.quantization.prepare(
    static_model,
    inplace=True
)


In [ ]:
with torch.no_grad():

    for images, _ in trainloader:

        static_model(images)

        break

In [ ]:
torch.quantization.convert(
    static_model,
    inplace=True
)

In [ ]:
static_acc = evaluate_accuracy(
    static_model,
    testloader
)

static_lat = measure_latency(
    static_model
)

static_size = model_size_mb(
    static_model
)

In [ ]:
qat_model = copy.deepcopy(
    fp32_model.cpu()
)

qat_model.train()

qat_model.qconfig = (
    torch.quantization.get_default_qat_qconfig("f

In [ ]:
torch.quantization.prepare_qat

In [ ]:
optimizer = optim.Adam(
    qat_model.parameters(),
    lr=1e-5
)

for epoch in range(1):

    for images, labels in trainloader:

        optimizer.zero_grad()

        outputs = qat_model

In [ ]:
qat_acc = evaluate_accuracy(
    qat_model,
    testloader
)

qat_lat = measure_latency(
    qat_model
)

qat_size = model_size_mb(
    qat_model
)

In [ ]:
results = {

    "FP32":
    [fp32_acc, fp32_lat, fp32_size],

    "Dynamic PTQ":
    [ptq_dyn_acc, ptq_dyn_lat, ptq_dyn_size],

    "Static

In [ ]:
def select_best(results):

    base_acc = results["FP32"][0]
    base_lat = results["FP32"][1]

    best = None
    best_score = -1

    for name, data in results.items():

        acc = data[0]
        lat = data[1]

        acc_drop = base_acc - acc

        if acc_drop > 0.02:
            continue

        latency_gain = base_lat / lat

        score = (
            0.7 * latency_gain +
            0.3 * acc
        )

        print(name, score)

        if score > best_score:
            best_score = score
            best = name

    return best

In [ ]:
best_model = select_best(results)

print("\nBEST MODEL =", best_model)

In [ ]:
names = list(results.keys())

accs = [results[x][0] for x in names]
lats = [results[x][1] for x in names]
sizes = [results[x][2] for x in names]

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(names, accs, marker='o')
plt.title("Accuracy")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(names, lats)
plt.title("Latency (ms)")
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.bar(names, sizes)
plt.title("Model Size (MB)")
plt.show()